# p00_setup — schemas, volume and libraries for the procurement work

Creates three new schemas next to the outcome project's `bronze` / `silver` / `gold` in catalog `dbw_wb_outcome`, plus a volume for raw API snapshots, then smoke-tests the libraries and both APIs. Safe to re-run: every step is idempotent, and nothing outside the three new schemas is created or changed.

%md-sandbox
<pre class="mermaid">
---
config:
  layout: elk
---
flowchart TD
    cat["dbw_wb_outcome"]
    old["bronze / silver / gold / ml<br/>outcome project (read only from here)"]
    pb["proc_bronze<br/>raw records as delivered by the APIs"]
    raw["raw (volume)<br/>untouched JSON pages, one folder per snapshot"]
    ps["proc_silver<br/>cleaned, typed, keyed model (contracts, suppliers, notices, ...)"]
    pg["proc_gold<br/>tender timelines and the knowledge graph (nodes / edges)"]
    cat --> old
    cat --> pb --> raw
    cat --> ps
    cat --> pg
</pre>
<script type="module">
  import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11.6.0/dist/mermaid.esm.min.mjs";
  import elkLayouts from "https://cdn.jsdelivr.net/npm/@mermaid-js/layout-elk@0.1.7/dist/mermaid-layout-elk.esm.min.mjs";
  mermaid.registerLayoutLoaders(elkLayouts);
  mermaid.initialize({ startOnLoad: true });
</script>

## 1. Schemas and volume

**What:** creates `proc_bronze`, `proc_silver`, `proc_gold` and the volume `proc_bronze.raw`.

**Why:** separate schemas keep the procurement work apart from the outcome project, so it can be rebuilt or dropped without touching the evaluated tables, while still living in the same catalog and joining to them directly. `IF NOT EXISTS` makes re-runs harmless.

In [ ]:
%sql
USE CATALOG dbw_wb_outcome;

CREATE SCHEMA IF NOT EXISTS proc_bronze COMMENT 'Procurement: raw contract awards and procurement notices as delivered by World Bank APIs';
CREATE SCHEMA IF NOT EXISTS proc_silver COMMENT 'Procurement: cleaned, typed, keyed data model (contracts, suppliers, notices, dimensions)';
CREATE SCHEMA IF NOT EXISTS proc_gold   COMMENT 'Procurement: tender timelines and knowledge-graph node/edge tables';

CREATE VOLUME IF NOT EXISTS proc_bronze.raw COMMENT 'Untouched API responses, one folder per source and snapshot date';

## 2. Libraries

**What:** installs `rapidfuzz` (fast string similarity) and `unidecode` (transliteration to ASCII); then restarts Python.

**Why:** the ML runtime already ships pandas, scikit-learn, scipy and requests, but not these two. Both are small, pure-wheel packages used by the entity-resolution code in `src/proc_intel`.

In [ ]:
%pip install -q rapidfuzz unidecode

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
dbutils.library.restartPython()

In [ ]:
%run ./proc_common

## 3. Smoke test

**What:** imports the shared package from `../src`, calls each API for one row, writes/reads/deletes a test file in the volume, and checks whether GraphFrames is available on this runtime.

**Why:** proves imports, internet access and volume permissions work *before* the long ingests run. The GraphFrames check decides how `p06_knowledge_graph` runs its multi-hop queries.

In [ ]:
import importlib.util
import rapidfuzz, unidecode, sklearn
from proc_intel.http import get_json

print("proc_intel loaded from", SRC_DIR)
print("rapidfuzz", rapidfuzz.__version__, "| scikit-learn", sklearn.__version__,
      "| unidecode check:", unidecode.unidecode("Côte d’Ivoire"))

awards = get_json(AWARDS_URL, {**AWARDS_PARAMS, "top": 1, "skip": 0})
print("awards API count:", awards["count"], "| as_of_date:", awards["data"][0]["as_of_date"])
notices = get_json(NOTICES_URL, {"format": "json", "rows": 1, "os": 0, "fl": "id,noticedate"})
print("notices API total:", notices["total"])

probe = f"{RAW_VOLUME}/_smoke_test.txt"
with open(probe, "w") as f:
    f.write("hello volume")
print(open(probe).read())
dbutils.fs.rm(probe)

print("graphframes python package:", importlib.util.find_spec("graphframes") is not None)

proc_intel loaded from /Workspace/Users/shabadvaswani2002@outlook.com/wb-procurement/src
rapidfuzz 3.14.6 | scikit-learn 1.6.1 | unidecode check: Cote d'Ivoire
awards API count: 295058 | as_of_date: 20-Oct-2026
notices API total: 423546
hello volume
graphframes python package: True
